# Curated Dish Dataset — Preparation and Cleaning

## Objective

This notebook focuses on creating a small curated dataset of food dishes with verified ingredient annotations.

Each dish will contain:
- A unique dish identifier.
- A dish name and food category.
- A list of ingredients.
- Ingredient quantities expressed in grams.
- A corresponding USDA FoodData Central identifier (`fdc_id`).

The curated dataset will connect food dishes with the nutritional reference information extracted from USDA Foundation Foods.

The objective is to establish a structured and traceable dataset that can later support ingredient-level nutritional estimation.

In [1]:

import pandas as pd
from pathlib import Path

# Project paths
cwd = Path.cwd()
PROJECT_ROOT = cwd.parent if cwd.name == "notebooks" else cwd

# Load processed USDA dataset
USDA_FILE = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "usda_foundation_nutrition.csv"
)

usda_df = pd.read_csv(USDA_FILE)

print("USDA DATASET OVERVIEW")
print("-" * 40)
print("Number of food records:", len(usda_df))
print("Columns:", list(usda_df.columns))

# Display food names and their identifiers
display(
    usda_df[
        ["fdc_id", "food_name"]
    ].sort_values("food_name").head(30)
)

# Search for potentially useful ingredients
keywords = [
    "chicken",
    "rice",
    "egg",
    "tomato",
    "potato",
    "onion",
    "beef",
    "fish",
    "milk",
    "cheese",
    "bread",
    "pasta"
]

for keyword in keywords:
    matches = usda_df[
        usda_df["food_name"].str.contains(
            keyword, case=False, na=False
        )
    ]

    print(f"\n{keyword.upper()} — {len(matches)} matches")

    display(
        matches[["fdc_id", "food_name"]].head(10)
    )


USDA DATASET OVERVIEW
----------------------------------------
Number of food records: 424
Columns: ['fdc_id', 'food_name', 'calories_kcal', 'protein_g', 'fat_g', 'carbohydrates_g', 'energy_nutrient_id', 'energy_method']


,fdc_id,food_name
394,2768188,"Alaska Pollock, raw"
175,2262074,"Almond butter, creamy"
157,2257045,"Almond milk, unsweetened, plain, refrigerated"
133,1999631,"Almond milk, unsweetened, plain, shelf stable"
417,2779262,"Amaranth, flour, pre-gelatinized, organic"
404,2779249,"Amaranth, puffed, organic"
338,2747652,"Anchovies, canned in olive oil, with salt, dra..."
141,2003590,"Apple juice, with added vitamin C, from concen..."
120,1750340,"Apples, fuji, with skin, raw"
121,1750341,"Apples, gala, with skin, raw"



CHICKEN — 10 matches


,fdc_id,food_name
29,331897,"Chicken, broilers or fryers, drumstick, meat o..."
30,331960,"Chicken, broiler or fryers, breast, skinless, ..."
222,2514746,"Chicken, ground, with additives, raw"
250,2646170,"Chicken, breast, boneless, skinless, raw"
251,2646171,"Chicken, thigh, boneless, skinless, raw"
314,2727566,"Chicken, drumstick, meat and skin, raw"
315,2727567,"Chicken, thigh, meat and skin, raw"
316,2727568,"Chicken, wing, meat and skin, raw"
317,2727569,"Chicken, breast, meat and skin, raw"
391,2759004,"Lunchmeat, chicken breast, sliced"



RICE — 14 matches


,fdc_id,food_name
39,334536,"Restaurant, Chinese, fried rice, without meat"
105,790214,"Flour, rice, white, unenriched"
113,1104812,"Flour, rice, brown"
114,1104867,"Flour, rice, glutinous"
217,2512380,"Rice, brown, long grain, unenriched, raw"
218,2512381,"Rice, white, long grain, unenriched, raw"
291,2710821,"Wild rice, dry, raw"
295,2710825,"Rice, black, unenriched, raw"
308,2710838,"Rice, red, unenriched, dry, raw"
403,2779248,"Brown rice, puffed, organic"


... (16 outputs left out)


BREAD — 5 matches


,fdc_id,food_name
9,324317,"Onion rings, breaded, par fried, frozen, prepa..."
104,790146,"Flour, bread, white, enriched, unbleached"
381,2758993,"Bread, white, commercial"
382,2758994,"Bread, 100% whole wheat, commercial"
383,2758995,"Bread, mulitgrain, commercial"



PASTA — 3 matches


,fdc_id,food_name
31,332282,"Sauce, pasta, spaghetti/marinara, ready-to-serve"
386,2758998,"Pasta, dry, enriched, spaghetti"
387,2759000,"Pasta, dry, whole grain, spaghetti"


## 2. Selecting and Validating USDA Ingredient References

Before creating the curated dish annotations, we identify suitable USDA food records for common ingredients.

Each ingredient must be associated with a specific USDA FoodData Central identifier (`fdc_id`).

We inspect the nutritional completeness of each selected record to avoid using unsuitable or incomplete references.

We also consider the food's physical state, such as raw, cooked, dried, or processed, to ensure consistency with the ingredient quantities used in our dish annotations.

In [2]:

# Candidate USDA food records
candidate_ids = [
    2646170,  # Chicken breast, boneless, skinless, raw
    2512381,  # White rice, long grain, raw
    748967,   # Whole egg, Grade A, large
    321360,   # Tomatoes, grape, raw
    790646,   # Yellow onions, raw
    2346401,  # Russet potatoes, raw
    2514743,  # Ground beef, 90% lean, raw
    2684441,  # Atlantic salmon, farm raised, raw
    2758998   # Spaghetti, dry, enriched
]

candidate_df = usda_df[
    usda_df["fdc_id"].isin(candidate_ids)
].copy()

# Check missing nutritional values
nutrient_columns = [
    "calories_kcal",
    "protein_g",
    "fat_g",
    "carbohydrates_g"
]

candidate_df["missing_nutrients"] = (
    candidate_df[nutrient_columns].isna().sum(axis=1)
)

# Display selected records
display(
    candidate_df[
        [
            "fdc_id",
            "food_name",
            "calories_kcal",
            "protein_g",
            "fat_g",
            "carbohydrates_g",
            "energy_method",
            "missing_nutrients"
        ]
    ].sort_values("food_name")
)

print("\nTOTAL CANDIDATES:", len(candidate_df))
print(
    "Candidates with all four nutritional values:",
    (candidate_df["missing_nutrients"] == 0).sum()
)


,fdc_id,food_name,calories_kcal,protein_g,fat_g,carbohydrates_g,energy_method,missing_nutrients
219,2514743,"Beef, ground, 90% lean meat / 10% fat, raw",190.0,18.20,12.80,0.00,Energy (Atwater Specific Factors),0
250,2646170,"Chicken, breast, boneless, skinless, raw",112.0,22.50,1.93,0.00,Energy (Atwater Specific Factors),0
97,748967,"Eggs, Grade A, Large, egg whole",148.0,12.40,9.96,0.96,Energy,0
264,2684441,"Fish, salmon, Atlantic, farm raised, raw",203.0,20.30,13.10,0.00,Energy (Atwater Specific Factors),0
109,790646,"Onions, yellow, raw",38.0,0.83,0.05,8.61,Energy,0
386,2758998,"Pasta, dry, enriched, spaghetti",371.0,15.10,1.85,73.50,Energy (Atwater General Factors),0
194,2346401,"Potatoes, russet, without skin, raw",81.0,2.27,0.36,17.80,Energy (Atwater Specific Factors),0
218,2512381,"Rice, white, long grain, unenriched, raw",370.0,7.04,1.03,80.30,Energy (Atwater Specific Factors),0
1,321360,"Tomatoes, grape, raw",27.0,0.83,0.63,5.51,Energy,0



TOTAL CANDIDATES: 9
Candidates with all four nutritional values: 9


In [3]:

# Create ingredient reference table

ingredient_reference = candidate_df[
    ["fdc_id", "food_name"]
].copy()

ingredient_reference = ingredient_reference.rename(
    columns={
        "fdc_id": "usda_fdc_id",
        "food_name": "usda_food_name"
    }
)

ingredient_reference.insert(
    0,
    "ingredient_id",
    [f"ING{i:03d}" for i in range(1, len(ingredient_reference) + 1)]
)

# Display the reference table
display(ingredient_reference)

# Create output directory
CURATED_DIR = (
    PROJECT_ROOT / "data" / "processed" / "curated"
)

CURATED_DIR.mkdir(parents=True, exist_ok=True)

# Save ingredient reference
ingredient_reference.to_csv(
    CURATED_DIR / "ingredient_reference.csv",
    index=False
)

print("Ingredient reference saved:")
print(CURATED_DIR / "ingredient_reference.csv")


,ingredient_id,usda_fdc_id,usda_food_name
1,ING001,321360,"Tomatoes, grape, raw"
97,ING002,748967,"Eggs, Grade A, Large, egg whole"
109,ING003,790646,"Onions, yellow, raw"
194,ING004,2346401,"Potatoes, russet, without skin, raw"
218,ING005,2512381,"Rice, white, long grain, unenriched, raw"
219,ING006,2514743,"Beef, ground, 90% lean meat / 10% fat, raw"
250,ING007,2646170,"Chicken, breast, boneless, skinless, raw"
264,ING008,2684441,"Fish, salmon, Atlantic, farm raised, raw"
386,ING009,2758998,"Pasta, dry, enriched, spaghetti"


Ingredient reference saved:
C:\Users\dell\NutriVision\data\processed\curated\ingredient_reference.csv


In [4]:

# Define the first 10 curated dishes

dishes_data = [
    ("DISH001", "Chicken with Rice", "Rice-based"),
    ("DISH002", "Beef with Rice", "Rice-based"),
    ("DISH003", "Salmon with Rice", "Rice-based"),
    ("DISH004", "Vegetable Omelette", "Egg-based"),
    ("DISH005", "Chicken Pasta", "Pasta"),
    ("DISH006", "Spaghetti with Ground Beef", "Pasta"),
    ("DISH007", "Baked Potato with Chicken", "Potato-based"),
    ("DISH008", "Tomato and Egg Dish", "Egg-based"),
    ("DISH009", "Chicken and Vegetable Bowl", "Mixed"),
    ("DISH010", "Beef and Potato Bowl", "Mixed"),
]

dishes_df = pd.DataFrame(
    dishes_data,
    columns=["dish_id", "dish_name", "category"]
)

display(dishes_df)

# Save dishes table
dishes_df.to_csv(
    CURATED_DIR / "dishes.csv",
    index=False
)

print("Dishes table saved:", CURATED_DIR / "dishes.csv")


,dish_id,dish_name,category
0,DISH001,Chicken with Rice,Rice-based
1,DISH002,Beef with Rice,Rice-based
2,DISH003,Salmon with Rice,Rice-based
3,DISH004,Vegetable Omelette,Egg-based
4,DISH005,Chicken Pasta,Pasta
5,DISH006,Spaghetti with Ground Beef,Pasta
6,DISH007,Baked Potato with Chicken,Potato-based
7,DISH008,Tomato and Egg Dish,Egg-based
8,DISH009,Chicken and Vegetable Bowl,Mixed
9,DISH010,Beef and Potato Bowl,Mixed


Dishes table saved: C:\Users\dell\NutriVision\data\processed\curated\dishes.csv


In [5]:

# Ingredient-level annotations for the first 10 dishes

ingredient_data = [
    # DISH001 - Chicken with Rice
    ("DISH001", "ING007", 150),
    ("DISH001", "ING005", 80),
    ("DISH001", "ING001", 50),
    ("DISH001", "ING003", 30),

    # DISH002 - Beef with Rice
    ("DISH002", "ING006", 150),
    ("DISH002", "ING005", 80),
    ("DISH002", "ING001", 50),
    ("DISH002", "ING003", 30),

    # DISH003 - Salmon with Rice
    ("DISH003", "ING008", 150),
    ("DISH003", "ING005", 80),
    ("DISH003", "ING001", 50),
    ("DISH003", "ING003", 30),

    # DISH004 - Vegetable Omelette
    ("DISH004", "ING002", 150),
    ("DISH004", "ING001", 60),
    ("DISH004", "ING003", 30),

    # DISH005 - Chicken Pasta
    ("DISH005", "ING007", 120),
    ("DISH005", "ING009", 80),
    ("DISH005", "ING001", 60),

    # DISH006 - Spaghetti with Ground Beef
    ("DISH006", "ING006", 120),
    ("DISH006", "ING009", 80),
    ("DISH006", "ING001", 60),

    # DISH007 - Baked Potato with Chicken
    ("DISH007", "ING004", 200),
    ("DISH007", "ING007", 120),
    ("DISH007", "ING003", 30),

    # DISH008 - Tomato and Egg Dish
    ("DISH008", "ING002", 150),
    ("DISH008", "ING001", 120),
    ("DISH008", "ING003", 30),

    # DISH009 - Chicken and Vegetable Bowl
    ("DISH009", "ING007", 150),
    ("DISH009", "ING004", 100),
    ("DISH009", "ING001", 60),
    ("DISH009", "ING003", 30),

    # DISH010 - Beef and Potato Bowl
    ("DISH010", "ING006", 150),
    ("DISH010", "ING004", 180),
    ("DISH010", "ING001", 50),
    ("DISH010", "ING003", 30),
]

ingredients_df = pd.DataFrame(
    ingredient_data,
    columns=["dish_id", "ingredient_id", "quantity_g"]
)

display(ingredients_df)

print("Total ingredient annotations:", len(ingredients_df))
print("Unique dishes:", ingredients_df["dish_id"].nunique())
print("Unique ingredients:", ingredients_df["ingredient_id"].nunique())


,dish_id,ingredient_id,quantity_g
0,DISH001,ING007,150
1,DISH001,ING005,80
2,DISH001,ING001,50
3,DISH001,ING003,30
4,DISH002,ING006,150
5,DISH002,ING005,80
6,DISH002,ING001,50
7,DISH002,ING003,30
8,DISH003,ING008,150
9,DISH003,ING005,80


Total ingredient annotations: 35
Unique dishes: 10
Unique ingredients: 9


In [7]:

# Validate dish references
invalid_dishes = set(ingredients_df["dish_id"]) - set(dishes_df["dish_id"])

# Validate ingredient references
invalid_ingredients = (
    set(ingredients_df["ingredient_id"])
    - set(ingredient_reference["ingredient_id"])
)

# Validate quantities
invalid_quantities = (
    ingredients_df["quantity_g"].isna().any()
    or (ingredients_df["quantity_g"] <= 0).any()
)

# Check duplicate dish-ingredient combinations
duplicate_pairs = ingredients_df.duplicated(
    subset=["dish_id", "ingredient_id"]
).sum()

# Display validation results
print("Invalid dish IDs:", invalid_dishes)
print("Invalid ingredient IDs:", invalid_ingredients)
print("Invalid quantities:", invalid_quantities)
print("Duplicate dish-ingredient pairs:", duplicate_pairs)

if (
    not invalid_dishes
    and not invalid_ingredients
    and not invalid_quantities
    and duplicate_pairs == 0
):
    print("\nAll validation checks passed!")
else:
    print("\nValidation failed. Review the issues above.")


Invalid dish IDs: set()
Invalid ingredient IDs: set()
Invalid quantities: False
Duplicate dish-ingredient pairs: 0

All validation checks passed!


In [8]:

# Save validated ingredient annotations

INGREDIENTS_FILE = CURATED_DIR / "ingredients.csv"

ingredients_df.to_csv(
    INGREDIENTS_FILE,
    index=False
)

print("Ingredient annotations saved successfully!")
print("File:", INGREDIENTS_FILE)
print("Total rows:", len(ingredients_df))


Ingredient annotations saved successfully!
File: C:\Users\dell\NutriVision\data\processed\curated\ingredients.csv
Total rows: 35


In [9]:

# Inspect the selected USDA nutritional references

nutrition_check = ingredient_reference.merge(
    usda_df,
    left_on="usda_fdc_id",
    right_on="fdc_id",
    how="left"
)

display(
    nutrition_check[
        [
            "ingredient_id",
            "usda_food_name",
            "calories_kcal",
            "protein_g",
            "fat_g",
            "carbohydrates_g",
            "energy_method"
        ]
    ]
)

print("Number of ingredient references:", len(nutrition_check))
print(
    "Missing nutritional values:",
    nutrition_check[
        ["calories_kcal", "protein_g", "fat_g", "carbohydrates_g"]
    ].isna().sum().sum()
)


,ingredient_id,usda_food_name,calories_kcal,protein_g,fat_g,carbohydrates_g,energy_method
0,ING001,"Tomatoes, grape, raw",27.0,0.83,0.63,5.51,Energy
1,ING002,"Eggs, Grade A, Large, egg whole",148.0,12.40,9.96,0.96,Energy
2,ING003,"Onions, yellow, raw",38.0,0.83,0.05,8.61,Energy
3,ING004,"Potatoes, russet, without skin, raw",81.0,2.27,0.36,17.80,Energy (Atwater Specific Factors)
4,ING005,"Rice, white, long grain, unenriched, raw",370.0,7.04,1.03,80.30,Energy (Atwater Specific Factors)
5,ING006,"Beef, ground, 90% lean meat / 10% fat, raw",190.0,18.20,12.80,0.00,Energy (Atwater Specific Factors)
6,ING007,"Chicken, breast, boneless, skinless, raw",112.0,22.50,1.93,0.00,Energy (Atwater Specific Factors)
7,ING008,"Fish, salmon, Atlantic, farm raised, raw",203.0,20.30,13.10,0.00,Energy (Atwater Specific Factors)
8,ING009,"Pasta, dry, enriched, spaghetti",371.0,15.10,1.85,73.50,Energy (Atwater General Factors)


Number of ingredient references: 9
Missing nutritional values: 0


In [11]:

import json

USDA_JSON = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "usda_foundation"
    / "FoodData_Central_foundation_food_json_2026-10-29.json"
)

with open(USDA_JSON, "r", encoding="utf-8") as file:
    usda_raw = json.load(file)

# Inspect one USDA food record: raw chicken breast
chicken_record = next(
    food for food in usda_raw["FoundationFoods"]
    if food["fdcId"] == 2646170
)

print("Food:", chicken_record["description"])
print("\nNutrient metadata:")

for nutrient in chicken_record["foodNutrients"]:
    nutrient_info = nutrient.get("nutrient", {})
    name = nutrient_info.get("name", "")

    if name in [
        "Energy",
        "Protein",
        "Total lipid (fat)",
        "Carbohydrate, by difference"
    ]:
        print(
            name,
            "| Amount:", nutrient.get("amount"),
            "| Unit:", nutrient_info.get("unitName")
        )


Food: Chicken, breast, boneless, skinless, raw

Nutrient metadata:
Total lipid (fat) | Amount: 1.93 | Unit: g
Protein | Amount: 22.5 | Unit: g
Carbohydrate, by difference | Amount: 0.0 | Unit: g


In [12]:

# Inspect the exact nutrient records used in our cleaned dataset

for nutrient in chicken_record["foodNutrients"]:
    nutrient_info = nutrient.get("nutrient", {})

    if nutrient_info.get("id") in [1008, 1003, 1004, 1005, 2047, 2048]:
        print(
            "ID:", nutrient_info.get("id"),
            "| Name:", nutrient_info.get("name"),
            "| Amount:", nutrient.get("amount"),
            "| Unit:", nutrient_info.get("unitName")
        )


ID: 1004 | Name: Total lipid (fat) | Amount: 1.93 | Unit: g
ID: 1003 | Name: Protein | Amount: 22.5 | Unit: g
ID: 1005 | Name: Carbohydrate, by difference | Amount: 0.0 | Unit: g
ID: 2047 | Name: Energy (Atwater General Factors) | Amount: 106 | Unit: kcal
ID: 2048 | Name: Energy (Atwater Specific Factors) | Amount: 112 | Unit: kcal


In [13]:

# Merge ingredient annotations with USDA references

ingredient_nutrition_df = (
    ingredients_df
    .merge(
        ingredient_reference,
        on="ingredient_id",
        how="left"
    )
    .merge(
        usda_df,
        left_on="usda_fdc_id",
        right_on="fdc_id",
        how="left"
    )
)

# Calculate nutritional contributions
nutrient_columns = [
    "calories_kcal",
    "protein_g",
    "fat_g",
    "carbohydrates_g"
]

for column in nutrient_columns:
    ingredient_nutrition_df[f"{column}_contribution"] = (
        ingredient_nutrition_df["quantity_g"] / 100
    ) * ingredient_nutrition_df[column]

# Display the results
display(
    ingredient_nutrition_df[
        [
            "dish_id",
            "ingredient_id",
            "usda_food_name",
            "quantity_g",
            "usda_fdc_id",
            "calories_kcal_contribution",
            "protein_g_contribution",
            "fat_g_contribution",
            "carbohydrates_g_contribution"
        ]
    ].round(2)
)

print("Total ingredient records:", len(ingredient_nutrition_df))
print(
    "Missing calculated values:",
    ingredient_nutrition_df[
        [
            "calories_kcal_contribution",
            "protein_g_contribution",
            "fat_g_contribution",
            "carbohydrates_g_contribution"
        ]
    ].isna().sum().sum()
)


,dish_id,ingredient_id,usda_food_name,quantity_g,usda_fdc_id,calories_kcal_contribution,protein_g_contribution,fat_g_contribution,carbohydrates_g_contribution
0,DISH001,ING007,"Chicken, breast, boneless, skinless, raw",150,2646170,168.0,33.75,2.90,0.00
1,DISH001,ING005,"Rice, white, long grain, unenriched, raw",80,2512381,296.0,5.63,0.82,64.24
2,DISH001,ING001,"Tomatoes, grape, raw",50,321360,13.5,0.42,0.32,2.76
3,DISH001,ING003,"Onions, yellow, raw",30,790646,11.4,0.25,0.02,2.58
4,DISH002,ING006,"Beef, ground, 90% lean meat / 10% fat, raw",150,2514743,285.0,27.30,19.20,0.00
5,DISH002,ING005,"Rice, white, long grain, unenriched, raw",80,2512381,296.0,5.63,0.82,64.24
6,DISH002,ING001,"Tomatoes, grape, raw",50,321360,13.5,0.42,0.32,2.76
7,DISH002,ING003,"Onions, yellow, raw",30,790646,11.4,0.25,0.02,2.58
8,DISH003,ING008,"Fish, salmon, Atlantic, farm raised, raw",150,2684441,304.5,30.45,19.65,0.00
9,DISH003,ING005,"Rice, white, long grain, unenriched, raw",80,2512381,296.0,5.63,0.82,64.24


Total ingredient records: 35
Missing calculated values: 0


In [14]:

# Define the nutritional contribution columns
contribution_columns = [
    "calories_kcal_contribution",
    "protein_g_contribution",
    "fat_g_contribution",
    "carbohydrates_g_contribution"
]

# Aggregate ingredient contributions by dish
dish_nutrition_df = (
    ingredient_nutrition_df
    .groupby("dish_id")
    .agg(
        total_ingredient_weight_g=("quantity_g", "sum"),
        calories_kcal=("calories_kcal_contribution", "sum"),
        protein_g=("protein_g_contribution", "sum"),
        fat_g=("fat_g_contribution", "sum"),
        carbohydrates_g=("carbohydrates_g_contribution", "sum")
    )
    .reset_index()
)

# Add dish names and categories
dish_nutrition_df = dishes_df.merge(
    dish_nutrition_df,
    on="dish_id",
    how="left"
)

# Round nutritional values
numeric_columns = [
    "calories_kcal",
    "protein_g",
    "fat_g",
    "carbohydrates_g"
]

dish_nutrition_df[numeric_columns] = (
    dish_nutrition_df[numeric_columns].round(2)
)

# Display final dish-level nutrition
display(dish_nutrition_df)

print("Total dishes:", len(dish_nutrition_df))
print(
    "Missing values:",
    dish_nutrition_df.isna().sum().sum()
)


,dish_id,dish_name,category,total_ingredient_weight_g,calories_kcal,protein_g,fat_g,carbohydrates_g
0,DISH001,Chicken with Rice,Rice-based,310,488.9,40.05,4.05,69.58
1,DISH002,Beef with Rice,Rice-based,310,605.9,33.60,20.35,69.58
2,DISH003,Salmon with Rice,Rice-based,310,625.4,36.75,20.80,69.58
3,DISH004,Vegetable Omelette,Egg-based,240,249.6,19.35,15.33,7.33
4,DISH005,Chicken Pasta,Pasta,260,447.4,39.58,4.17,62.11
5,DISH006,Spaghetti with Ground Beef,Pasta,260,541.0,34.42,17.22,62.11
6,DISH007,Baked Potato with Chicken,Potato-based,350,307.8,31.79,3.05,38.18
7,DISH008,Tomato and Egg Dish,Egg-based,300,265.8,19.85,15.71,10.63
8,DISH009,Chicken and Vegetable Bowl,Mixed,340,276.6,36.77,3.65,23.69
9,DISH010,Beef and Potato Bowl,Mixed,410,455.7,32.05,20.18,37.38


Total dishes: 10
Missing values: 0


In [15]:

# Validate dish-level nutritional estimates

nutrition_columns = [
    "calories_kcal",
    "protein_g",
    "fat_g",
    "carbohydrates_g"
]

print("Number of dishes:", len(dish_nutrition_df))
print("Unique dish IDs:", dish_nutrition_df["dish_id"].nunique())

print(
    "Missing nutritional values:",
    dish_nutrition_df[nutrition_columns].isna().sum().sum()
)

print(
    "Negative nutritional values:",
    (dish_nutrition_df[nutrition_columns] < 0).sum().sum()
)

print(
    "Invalid ingredient weights:",
    (dish_nutrition_df["total_ingredient_weight_g"] <= 0).sum()
)

# Save only if all checks pass
if (
    len(dish_nutrition_df) == 10
    and dish_nutrition_df["dish_id"].nunique() == 10
    and not dish_nutrition_df[nutrition_columns].isna().any().any()
    and not (dish_nutrition_df[nutrition_columns] < 0).any().any()
    and not (dish_nutrition_df["total_ingredient_weight_g"] <= 0).any()
):
    output_file = CURATED_DIR / "dish_nutrition.csv"

    dish_nutrition_df.to_csv(output_file, index=False)

    print("\nAll validation checks passed!")
    print("Saved:", output_file)
else:
    print("\nValidation failed. Review the results.")


Number of dishes: 10
Unique dish IDs: 10
Missing nutritional values: 0
Negative nutritional values: 0
Invalid ingredient weights: 0

All validation checks passed!
Saved: C:\Users\dell\NutriVision\data\processed\curated\dish_nutrition.csv
